# Mem0 记忆系统入门

使用 **DeepSeek** 作为 LLM + **Qdrant 本地模式** 作为向量数据库，构建 AI 记忆系统。

## 环境依赖
在终端运行以下命令安装依赖：

In [ ]:
# 在终端运行：
# pip install mem0ai qdrant-client sentence-transformers

## 1. 导入模块 & 配置 API 密钥

In [3]:
import os
from mem0 import Memory

# ==========================================
# 请替换为您真实的 DeepSeek API Key
# DeepSeek 控制台: https://platform.deepseek.com/api_keys
# ==========================================
os.environ["OPENAI_API_KEY"] = "sk-330f857531c74add872470803624889f"

## 2. 配置 Mem0（LLM + Embedding + 向量数据库）

In [4]:
# ==========================================
# Mem0 配置说明:
# - LLM:      DeepSeek Chat（兼容 OpenAI 接口）
# - Embedder: sentence-transformers（免费本地模型）
# - 向量库:    Qdrant 本地文件模式
# ==========================================
config = {
    "llm": {
        "provider": "openai",
        "config": {
            "model": "deepseek-chat",
            "openai_base_url": "https://api.deepseek.com/v1",
        },
    },
    "embedder": {
        "provider": "huggingface",
        "config": {
            "model": "sentence-transformers/all-MiniLM-L6-v2",
            "embedding_dims": 384,
        },
    },
    "vector_store": {
        "provider": "qdrant",
        "config": {
            "collection_name": "quickstart_mem0",
            "embedding_model_dims": 384,
            "path": "./qdrant_db",
            "on_disk": True,
        },
    },
    "version": "v1.1",
}

# 初始化 Memory 实例
m = Memory.from_config(config)
print("✅ Memory 初始化成功！")

Skipping import of cpp extensions due to incompatible torch version 2.11.0+cpu for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0801 18:08:17.917000 56504 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
d:\CondaData\envs\llm-finetune\lib\site-packages\torchao\dtypes\floatx\__init__.py:5: DeprecationWarning: Importing from torchao.dtypes.floatx.floatx_tensor_core_layout is deprecated. Please use 'from torchao.prototype.dtypes.floatx.floatx_tensor_core_layout import ...' instead. This import path will be removed in a future torchao release. Please check issue: https://github.com/pytorch/ao/issues/2752 for more details. 
  from .floatx_tensor_core_layout import (
d:\CondaData\envs\llm-finetune\lib\site-packages\torchao\dtypes\uintx\__init__.py:1: DeprecationWarning: Importing from torchao.dtypes.uintx.dyn_int8_act_int4_wei_cpu_layout is deprec

✅ Memory 初始化成功！


## 3. 添加记忆 (Add Memory)

为不同用户添加记忆，每条记忆可以附带元数据（metadata）便于分类筛选。

In [6]:
# 添加 Alice 的第一条记忆 —— 兴趣爱好
res1 = m.add(
    messages="Working on improving my tennis skills.",
    user_id="alice",
    metadata={"category": "hobbies"},
)
print("📝 添加 hobbies 记忆结果：")
print(res1, "\n")

# 添加 Alice 的第二条记忆 —— 待办任务
res2 = m.add(
    messages="I have a linear algebra midterm exam on November 20",
    user_id="alice",
    metadata={"category": "task"},
)
print("📝 添加 task 记忆结果：")
print(res2)

📝 添加 hobbies 记忆结果：
{'results': []} 

📝 添加 task 记忆结果：
{'results': []}


## 4. 获取所有记忆 (Get All Memories)

通过 `filters` 参数按用户 ID 筛选指定用户的所有记忆。

In [7]:
# 获取 Alice 的所有记忆
all_memories = m.get_all(filters={"user_id": "alice"})
print(all_memories)

{'results': [{'id': '0ea67eab-6d13-4330-a86b-31b7d1c5ec83', 'memory': 'User has a linear algebra midterm exam on November 20, 2026.', 'hash': '083a54f0d36e176148e46e086fce3532', 'metadata': {'category': 'task'}, 'created_at': '2026-08-01T10:07:24.270495+00:00', 'updated_at': '2026-08-01T10:07:24.270495+00:00', 'user_id': 'alice', 'attributed_to': 'user'}, {'id': 'af596d04-44cb-45fe-9fe4-2ee0da9cc6fd', 'memory': 'User is working on improving their tennis skills.', 'hash': 'b0ca27a409e66e0febe4b8219cce8ece', 'metadata': {'category': 'hobbies'}, 'created_at': '2026-08-01T10:07:23.119905+00:00', 'updated_at': '2026-08-01T10:07:23.119905+00:00', 'user_id': 'alice', 'attributed_to': 'user'}]}


## 5. 语义搜索记忆 (Search Memory)

用自然语言描述查询内容，Mem0 会自动做**语义相似度搜索**，找出最相关的记忆。

> `score` 越高表示越相关（L2 距离转换后的相似度得分）。

In [ ]:
search_results = m.search(
    query="What are Alice's hobbies",
    filters={"user_id": "alice"},
)
print("🔍 搜索结果：")
print(search_results)

Failed to load spaCy lemma model: spaCy is not installed. Install it with: pip install mem0ai[nlp]
Failed to load spaCy full model: spaCy is not installed. Install it with: pip install mem0ai[nlp]
fastembed not installed - BM25 keyword search disabled. Install it with: pip install "mem0ai[extras]"


🔍 搜索结果：
{'results': [{'id': 'af596d04-44cb-45fe-9fe4-2ee0da9cc6fd', 'memory': 'User is working on improving their tennis skills.', 'hash': 'b0ca27a409e66e0febe4b8219cce8ece', 'metadata': {'category': 'hobbies'}, 'score': 0.1457772826132579, 'created_at': '2026-08-01T10:07:23.119905+00:00', 'updated_at': '2026-08-01T10:07:23.119905+00:00', 'user_id': 'alice', 'attributed_to': 'user'}, {'id': '0ea67eab-6d13-4330-a86b-31b7d1c5ec83', 'memory': 'User has a linear algebra midterm exam on November 20, 2026.', 'hash': '083a54f0d36e176148e46e086fce3532', 'metadata': {'category': 'task'}, 'score': 0.13022264702543068, 'created_at': '2026-08-01T10:07:24.270495+00:00', 'updated_at': '2026-08-01T10:07:24.270495+00:00', 'user_id': 'alice', 'attributed_to': 'user'}]}


: 

## 📊 总结

| 组件 | 选型 | 说明 |
|------|------|------|
| **LLM** | DeepSeek Chat | 兼容 OpenAI 接口，通过 `openai_base_url` 切换 |
| **Embedding** | all-MiniLM-L6-v2 | HuggingFace 本地模型，384 维，免费 |
| **向量库** | Qdrant 本地模式 | 持久化存储到 `./qdrant_db/`，无需额外服务 |

### 核心 API

```python
# 添加记忆
m.add(messages="...", user_id="...", metadata={...})

# 获取所有记忆
m.get_all(filters={"user_id": "..."})

# 语义搜索
m.search(query="...", filters={"user_id": "..."})
```